In [20]:
from langgraph.graph import StateGraph,START,END
from dotenv import load_dotenv
from typing import TypedDict,Annotated
from langchain_groq import ChatGroq
from pydantic import BaseModel,Field
import operator

In [21]:
load_dotenv()

model= ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.3,
)

In [22]:
class Evaluationschema(BaseModel):
    feedback : str=Field(description="detailed feedback for the essay")
    score: int = Field(description="score out of 10", ge=0, le=10)

In [23]:
structured_model=model.with_structured_output(Evaluationschema)

In [24]:
essay="""The Importance of Time Management

Time is one of the most valuable resources available to human beings. Unlike money or material possessions, time cannot be earned back once it is lost. Therefore, managing time effectively is essential for achieving success, maintaining productivity, and leading a balanced life.

Time management involves planning and organizing tasks in a way that allows individuals to make the best use of their available hours. People who manage their time well are often able to complete their responsibilities efficiently while still having time for relaxation and personal interests. On the other hand, poor time management can lead to stress, missed deadlines, and reduced performance.

For students, time management is particularly important. Balancing classes, assignments, projects, and extracurricular activities requires careful planning. By creating schedules and setting priorities, students can avoid last-minute pressure and improve their academic performance. Similarly, professionals rely on time management to meet deadlines, achieve goals, and maintain a healthy work-life balance.

Technology can be both a help and a hindrance to time management. Productivity tools, calendars, and reminder applications can help individuals stay organized. However, excessive use of social media and entertainment platforms can become major distractions. Learning to control these distractions is a crucial part of managing time effectively.

Good time management also contributes to personal growth. It allows individuals to dedicate time to learning new skills, exercising, pursuing hobbies, and spending quality time with family and friends. As a result, people become more productive, confident, and satisfied with their lives.

In conclusion, time management is a vital skill that benefits every aspect of life. By planning wisely, setting priorities, and avoiding distractions, individuals can make the most of their time and work toward their goals more effectively. Success is often not about having more time, but about using the time available in the best possible way."""

In [25]:
ans=structured_model.invoke(f"evaluate the language quality of the essay and provide a feedback and a score out of 10 {essay}")

In [26]:
ans.feedback

'The essay provides a clear and well-structured argument on the importance of time management. It effectively uses examples and explanations to support its claims. However, some sentences are wordy and could be rephrased for better clarity. The essay could also benefit from more specific examples and anecdotes to make it more engaging. Overall, the language quality is good, but there is room for improvement.'

In [27]:
ans.score

8

In [28]:
class UPSCstate(TypedDict):
    essay : str
    language_feedback : str
    analysis_feedback :str
    clarity_feedback :str
    overall_feedback :str
    individual_scores : Annotated[list[int],operator.add]
    avg_score : float

In [29]:
graph=StateGraph(UPSCstate)

In [30]:
def evaluate_language(state:UPSCstate):
    prompt = f"evaluate the language quality of the essay and provide a feedback and a score out of 10 {state['essay']}"
    output=structured_model.invoke(prompt)

    return {'language_feedback':output.feedback,
            'individual_scores':[output.score]}   

In [31]:
def evaluate_analysis(state:UPSCstate):
    prompt = f"evaluate the analysis quality of the essay and provide a feedback and a score out of 10 {state['essay']}"
    output=structured_model.invoke(prompt)

    return {'analysis_feedback':output.feedback,
            'individual_scores':[output.score]}   

In [32]:
def evaluate_thought(state:UPSCstate):
    prompt = f"evaluate the clarity of thought quality of the essay and provide a feedback and a score out of 10 {state['essay']}"
    output=structured_model.invoke(prompt)

    return {'clarity_feedback':output.feedback,
            'individual_scores':[output.score]}   

In [33]:
def final_evaluation(state:UPSCstate):
    # summarized feedback
    prompt = f"""Based on the following feedbacks create a summarized feedback
        \n language feedback - {state["language_feedback"]}
        \n depth of analysis feedback - {state["analysis_feedback"]}
        \n clarity of thought feedback - {state["clarity_feedback"]}"""
    
    overall_feedback=model.invoke(prompt).content

    # avg score
    avg_score=sum(state['individual_scores'])/len(state['individual_scores'])

    return {'overall_feedback':overall_feedback,
            'avg_score':avg_score}

In [34]:
graph.add_node('evaluate_language',evaluate_language)
graph.add_node('evaluate_analysis',evaluate_analysis)
graph.add_node('evaluate_thought',evaluate_thought)
graph.add_node('final_evaluation',final_evaluation)

In [35]:
graph.add_edge(START,'evaluate_language')
graph.add_edge(START,'evaluate_analysis')
graph.add_edge(START,'evaluate_thought')

graph.add_edge('evaluate_language','final_evaluation')
graph.add_edge('evaluate_analysis','final_evaluation')
graph.add_edge('evaluate_thought','final_evaluation')

graph.add_edge('final_evaluation',END)

In [36]:
workflow=graph.compile()

In [37]:
initial_state={'essay':essay}
workflow.invoke(initial_state)


{'essay': 'The Importance of Time Management\n\nTime is one of the most valuable resources available to human beings. Unlike money or material possessions, time cannot be earned back once it is lost. Therefore, managing time effectively is essential for achieving success, maintaining productivity, and leading a balanced life.\n\nTime management involves planning and organizing tasks in a way that allows individuals to make the best use of their available hours. People who manage their time well are often able to complete their responsibilities efficiently while still having time for relaxation and personal interests. On the other hand, poor time management can lead to stress, missed deadlines, and reduced performance.\n\nFor students, time management is particularly important. Balancing classes, assignments, projects, and extracurricular activities requires careful planning. By creating schedules and setting priorities, students can avoid last-minute pressure and improve their academic